# DevOps, HA/DR & Migration

The last three operational pillars sit together for a reason: how you *ship* code (DevOps), how you *survive* failure (HA/DR), and how you *get there from here* (migration) are tightly coupled. A robust deployment pipeline is what lets you exercise failover regularly; a tested DR posture is what makes migrations safe; a disciplined migration is the first test of both.

This notebook covers the spine: **Azure DevOps and GitHub Actions** as the two pipeline runtimes, **Bicep and ARM** for infrastructure as code, **deployment strategies** for safe rollouts, **Azure Backup and Site Recovery** for data and workload protection, **HA and DR patterns** for the design, and **Azure Migrate plus DMS plus Data Box** for the move-in journey.

## Azure DevOps vs GitHub Actions

Microsoft owns two CI/CD platforms; both are first-class on Azure and both are here to stay.

**Azure DevOps Services** bundles five products under one tenant:

- **Repos** — git hosting.
- **Pipelines** — YAML-defined CI/CD with hosted Linux/Windows/Mac agents and the ability to self-host.
- **Boards** — work items, sprints, Kanban.
- **Artifacts** — package feeds (NuGet, npm, Maven, PyPI, generic).
- **Test Plans** — manual test management.

**GitHub Actions on Azure** is the modern direction for most teams — code lives in GitHub, Actions runs CI/CD, deploys to Azure via the `azure/login` action with OIDC federation. The connector ecosystem is broader, the community is the world's, and Microsoft is shipping Azure-specific actions on Actions first.

When to pick which:

- **Enterprise already in Azure DevOps** with Boards integrated to Pipelines and on-prem TFVC migrations → keep using Azure DevOps Pipelines.
- **New team / open-source-style flow / multi-cloud or cross-org** → GitHub + Actions.
- **Mixed estate** — many shops run both, with Pipelines on legacy projects and Actions on new ones.

Both speak YAML pipelines, support reusable templates, and integrate with Azure for deployments. Mechanically, you can swap between them with modest effort.

## Service connections and OIDC federation

However you run the pipeline, the hard problem is: *how does the runner authenticate to Azure to deploy?* The wrong answer (and the legacy default) is **a client secret on an Entra ID app registration, stored in the pipeline as a secret variable**. Secrets leak, rotate poorly, and live longer than they should.

The right answer is **federated credentials / OIDC**:

- On the Entra ID app registration, you add a **federated credential** that trusts a specific GitHub repo + branch (or Azure DevOps service connection).
- At pipeline run time, the runner gets a short-lived OIDC token signed by GitHub/Azure DevOps.
- It exchanges the OIDC token for an Entra ID access token via Workload Identity Federation. No secrets stored anywhere.

In **GitHub Actions**, the `azure/login` action with `client-id` + `tenant-id` + `subscription-id` and an `id-token: write` permission does this end to end.

In **Azure DevOps**, configure a **service connection** of type **"Workload Identity federation"** instead of "Service principal (manual)".

Combine OIDC federation with a **least-privilege role assignment** on the target scope (Contributor on one resource group, not Owner on the subscription), and you have a deployment identity that is short-lived, scoped, and auditable. This is the modern best practice; treat anything else as legacy.

## Bicep and ARM templates

Infrastructure as code on Azure has two first-party options:

- **ARM templates** — JSON, declarative, the original. Verbose, harder to read.
- **Bicep** — a DSL that transpiles to ARM. Cleaner syntax, modules, type checking, `az deployment` integration. Microsoft's recommended IaC tool for Azure-only deployments.

Most teams either use **Bicep** (Azure-native, less ceremony) or **Terraform with the AzureRM provider** (multi-cloud, ecosystem, state management). Pulumi and Crossplane have niches but smaller share.

Key Bicep features worth knowing:

- **Modules** — reusable units, parameterised, composable. The right unit of sharing across teams.
- **What-if** — `az deployment group what-if` shows exactly what will change before you apply. Use it in CI as a guard rail.
- **Deployment scopes** — resource group, subscription, management group, tenant. The scope shapes which resource types you can deploy.
- **Deployment Stacks** — newer feature that bundles a deployment with **deny assignments** so the deployed resources can't be modified out-of-band; supersedes Azure Blueprints.
- **Template Specs** — versioned, shareable templates stored in a resource group; teams reference them as a module source.

AWS comparison: Bicep ≈ CloudFormation's domain (Azure-native IaC); Terraform usage is essentially identical on both clouds.

## Deployment strategies

Four ways to roll out a new version, in order of risk:

- **In-place** — overwrite the running instance with the new version. Fastest, riskiest. Acceptable for stateless backend updates and small deployments.
- **Rolling** — update N% of instances at a time, wait for health to confirm, continue. The VM Scale Set rolling upgrade pattern from notebook 03.
- **Blue-green** — provision a new environment (blue) alongside production (green), test it, swap routing. App Service **deployment slots** are this pattern in one click — swap is near-instant and reversible.
- **Canary** — route a small percentage of real traffic to the new version, watch metrics, gradually increase. Container Apps' **revisions with weighted traffic split** and Front Door's **rules-engine percentage routing** both support canary natively.

**Pick the strategy that the service supports cheaply**. App Service → slots (blue-green). Container Apps → revisions (canary). VMSS → rolling upgrades. AKS → use a deployment controller (Argo Rollouts, Flagger) or Linkerd/Istio traffic shifting.

Always pair a strategy with **rollback automation** — health probes that fail the deployment, a one-click revert (slot swap back, revision weight reset). The strategy that needs a human to roll it back at 3 a.m. is a strategy that will not get rolled back at 3 a.m.

## Azure Backup

**Azure Backup** is the managed backup service. The model:

- A **Recovery Services vault** (classic) or **Backup vault** (newer, for Azure Files, blobs, disks, PostgreSQL) holds the backups.
- **Backup policies** define schedule, retention, and instant-restore window.
- **Soft delete** retains deleted backups for 14 days minimum; **immutable vaults** lock policies against tampering (a ransomware defence — even an Owner can't shorten retention or delete the vault).

What it backs up:

- **VMs** — app-consistent snapshots via the VM Backup extension (volume shadow copy on Windows, scripts on Linux).
- **SQL on VM** — log-shipping with point-in-time restore down to seconds.
- **SAP HANA on VM** — supported with proper extensions.
- **Azure Files / Blob / Disks** — vault-based snapshots.
- **PostgreSQL Flexible Server**, **Azure Database for PostgreSQL Single Server**, etc.
- **On-prem workloads** via MARS (Microsoft Azure Recovery Services) agent or MABS (System Center DPM in cloud).

**Backup is not DR**. It is *point-in-time recovery* for accidental deletes, ransomware, or corruption. The RTO is hours, the RPO is the schedule interval. For lower RTO/RPO, you need **Azure Site Recovery**.

## Azure Site Recovery

**Azure Site Recovery (ASR)** replicates workloads to a secondary region (or from on-prem to Azure) and orchestrates failover. The model:

- A **Recovery Services vault** in the *target* region holds replication state.
- For each protected VM, ASR replicates disks asynchronously to the target — typically achieving an **RPO of seconds to a few minutes**.
- **Replication groups** + **recovery plans** orchestrate multi-VM, multi-tier failovers in dependency order (DB before app before web).
- **Test failover** boots a copy in an isolated network to validate without affecting production — practise it quarterly.
- **Planned failover** is a clean swap when you know an outage is coming.
- **Unplanned failover** is the actual disaster path; recovery point is the latest replicated state.

ASR is also Microsoft's preferred path for *VM migration* into Azure: install the ASR agent on-prem, let it replicate to Azure, then "fail over" as the cutover. (More on migration below.)

AWS comparison: Site Recovery ≈ Elastic Disaster Recovery (DRS); Backup ≈ AWS Backup.

## HA and DR patterns

The same patterns apply across services; the implementation differs by tier.

- **Zonal** — pin a single VM/instance to one zone. Failure of that zone takes it down.
- **Zone-redundant** — deploy across all three zones in the region; survives one-zone failure. The default for production. Available for VMSS (Flexible), App Service Premium v3, App Gateway v2, Standard Load Balancer, ZRS storage, AZ-redundant SQL.
- **Active-passive multi-region** — primary region serves all traffic; secondary region holds replicated state and is brought online on failover. Cheapest multi-region; RTO measured in minutes to hours depending on automation.
- **Active-active multi-region** — both regions serve traffic; replication is bidirectional or via a shared store. Lowest RTO/RPO; highest cost and complexity. Front Door + multi-region App Gateways + Cosmos DB multi-write + Azure SQL active geo-replication is the canonical stack.

**RTO (Recovery Time Objective)** and **RPO (Recovery Point Objective)** are the two numbers that drive every DR conversation:

- RTO = how long you can be down.
- RPO = how much data you can afford to lose.

Quote them per workload, not per organisation. The customer-facing transactional service may need RTO < 5 min, RPO < 30 sec; the back-office reporting database may be fine with RTO 24 h, RPO 4 h. Buying multi-region active-active for everything is the costliest mistake in DR planning.

## Migration — Azure Migrate, DMS, Data Box

Moving an existing estate to Azure has its own product family.

**Azure Migrate** is the umbrella hub:

- **Discovery & assessment** — appliances installed in the source environment (VMware, Hyper-V, physical, AWS, GCP) inventory the workloads, measure utilisation, and produce sizing + cost estimates for Azure equivalents.
- **Server Migration** — replicate and cut over to Azure VMs, powered by ASR underneath.
- **Database assessment** — Data Migration Assistant evaluates compatibility for SQL Server moves.
- **Web App migration** — App Service Migration Assistant moves IIS / Java web apps.

**Azure Database Migration Service (DMS)** handles the actual database cutover: SQL Server → Azure SQL DB / MI, MySQL → Azure Database for MySQL, PostgreSQL → Flexible Server, Mongo → Cosmos DB. Two modes:

- **Offline** — single cutover; downtime equals the data copy time.
- **Online** — initial copy + continuous CDC replication; cutover is a near-zero-downtime switch.

**Azure Data Box family** handles bulk data transfer when bandwidth would make a network upload impractical:

- **Data Box** — 80 TB physical appliance shipped to you, you load data, you ship back. Microsoft ingests into your storage account.
- **Data Box Disk** — 8×8 TB SSD packs for smaller jobs.
- **Data Box Heavy** — 800 TB on wheels.
- **Data Box Gateway** — virtual appliance that exposes SMB/NFS locally and uploads asynchronously over your existing link.

Rule of thumb: if shipping a Data Box is faster than your internet link can sustain over the migration window, ship the box.

AWS comparison: Azure Migrate ≈ AWS Application Migration Service + Migration Hub; DMS ≈ AWS DMS; Data Box ≈ AWS Snowball/Snowmobile.

In [ ]:
# A Bicep + GitHub Actions OIDC flow, conceptually.

# main.bicep
cat <<'BICEP' > main.bicep
param location string = resourceGroup().location
param appName string

resource plan 'Microsoft.Web/serverfarms@2023-01-01' = {
  name: 'plan-${appName}'
  location: location
  sku: { name: 'P1v3', tier: 'PremiumV3' }
  kind: 'linux'
  properties: { reserved: true }
}

resource app 'Microsoft.Web/sites@2023-01-01' = {
  name: appName
  location: location
  properties: { serverFarmId: plan.id, httpsOnly: true }
}
BICEP

# .github/workflows/deploy.yaml (excerpt)
cat <<'YML' > deploy.yaml
permissions:
  id-token: write
  contents: read
jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: azure/login@v2
        with:
          client-id:        ${{ secrets.AZURE_CLIENT_ID }}     # app registration
          tenant-id:        ${{ secrets.AZURE_TENANT_ID }}
          subscription-id:  ${{ secrets.AZURE_SUBSCRIPTION_ID }}
      - run: |
          az deployment group what-if -g rg-app-prod -f main.bicep -p appName=myapp
          az deployment group create  -g rg-app-prod -f main.bicep -p appName=myapp
YML

# No client secret anywhere. The federated credential on the app registration
# trusts this repo + branch; tokens are minted per run.

## Putting it together

A modern Azure delivery + resilience stack:

1. **Source of truth** — GitHub (or Azure Repos), with the landing zone in Bicep, applications in their own repos with Bicep modules for their resources.
2. **Pipelines** — GitHub Actions (preferred) or Azure DevOps Pipelines, authenticating via OIDC federation; least-privilege role assignments per environment scope.
3. **Deploy strategy** — App Service slot swap, Container Apps revision split, or VMSS rolling upgrade depending on service. Health probes wired to fail bad deploys automatically.
4. **What-if in CI** — every PR runs `az deployment what-if`; reviewers see exactly what will change before merge.
5. **Backup** — Azure Backup with immutable vault for VMs, databases, and Files; quarterly restore drills.
6. **DR** — zone-redundant by default; multi-region active-passive (or active-active) for the customer-facing tier with explicit RTO/RPO targets; ASR for VM-based DR; cross-region geo-replication / auto-failover groups for managed databases.
7. **Migration** — for any move-in, Azure Migrate to discover and assess, ASR / Server Migration for VMs, DMS for databases, Data Box for bulk data; everything lands in the same IaC that the existing estate uses.

The discipline ties together: code is shipped through pipelines that prove they can deploy *and* roll back; failure is rehearsed through periodic failover and restore drills; migration arrives into the same shape, so the moved workload behaves like every other Azure workload from day one.